In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
import os
import pickle

import sys
sys.path.append("/home/z5297792/ESP_zonodo")
from functions import doppio, out_core_param_fit

sys.path.append('/home/z5297792/UNSW-MRes/MRes/SEACOFS_dataset') 
from clim_functions import doppio_pipeliner, find_directional_radii


In [2]:
fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_01461.nc'

dataset = nc.Dataset(fname)
lon_rho = np.transpose(dataset.variables['lon_rho'], axes=(1, 0))
lat_rho = np.transpose(dataset.variables['lat_rho'], axes=(1, 0))
mask_rho = np.transpose(dataset.variables['mask_rho'], axes=(1, 0))
h = np.transpose(dataset.variables['h'], axes=(1, 0))
# f = np.transpose(dataset.variables['f'], axes=(1, 0))
angle = dataset.variables['angle'][0, 0]
z_r = np.load('/srv/scratch/z5297792/z_r.npy')
z_r = np.transpose(z_r, (1, 2, 0))
def distance(lat1, lon1, lat2, lon2):
    EARTH_RADIUS = 6357
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
j_mid, i_mid = lon_rho.shape[1] // 2, lon_rho.shape[0] // 2
dx = distance(lat_rho[:-1, j_mid], lon_rho[:-1, j_mid],
              lat_rho[1:, j_mid], lon_rho[1:, j_mid])
dy = distance(lat_rho[i_mid, :-1], lon_rho[i_mid, :-1],
              lat_rho[i_mid, 1:], lon_rho[i_mid, 1:])
x_grid = np.insert(np.cumsum(dx), 0, 0)
y_grid = np.insert(np.cumsum(dy), 0, 0)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid, indexing='ij')


In [3]:
df_doppio = pd.read_pickle('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/df_DOPPIO_SEACOFS_26yr.pkl')
df_doppio['w'] *= 1e-3; df_doppio['Omega0'] *= 1e-3; df_doppio['Omega'] *= 1e-3; #df_doppio['psi0'] *= 1e9
df_doppio


,Day,fnumber,nxc,nyc,nCyc,xc,yc,w,Q,Omega0,Omega,Rc,psi0,R
0,1462,1461,830.0,1515.0,AE,829.879126,1515.297884,0.000016,"[[0.7054724746035225, -0.17299518294950286], [...",0.000007,0.000008,96.456316,-38.462785,63.384028
1,1462,1461,358.0,1408.0,AE,357.902634,1407.794571,0.000032,"[[1.2525415454561337, -0.37318304544145586], [...",0.000015,0.000014,78.042405,-41.260426,50.068171
2,1462,1461,928.0,1356.0,CE,928.181752,1356.070550,-0.000011,"[[0.9184334895938384, -0.6568144522261018], [-...",-0.000004,-0.000007,118.032741,47.992205,74.597458
3,1462,1461,506.0,1354.0,CE,505.750855,1353.499933,-0.000032,"[[1.0915204469734063, -0.17483272531676586], [...",-0.000016,-0.000013,106.666574,75.106895,69.457705
4,1462,1461,754.0,1285.0,AE,753.849693,1284.689807,0.000022,"[[1.300245004042572, -0.37537759847059776], [-...",0.000010,0.000008,103.327517,-44.066805,65.961546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416246,10650,10641,349.0,158.0,AE,348.663439,158.138527,0.000033,"[[1.3863808771116255, 0.04649055064735413], [0...",0.000016,0.000012,110.552001,-72.997132,43.099927
416247,10650,10641,973.0,126.0,CE,978.294053,127.970950,-0.000004,"[[0.6233322936879349, -0.5502050858849237], [-...",-0.000001,-0.000002,75.127332,4.241051,34.482872
416248,10650,10641,805.0,95.0,AE,804.765913,95.382012,0.000011,"[[1.2619918374118362, 0.3980497594289241], [0....",0.000005,0.000006,60.426625,-10.779905,45.893895
416249,10650,10641,157.0,34.0,CE,157.210669,33.449590,-0.000007,"[[0.9874338034387458, 0.628992620695803], [0.6...",-0.000003,-0.000004,141.142981,36.082072,34.764443


In [4]:
def clean_doppio(df_doppio, X_grid, Y_grid, threshold=4, Omega0_thresh=5e-5):

    required = ['nCyc', 'xc', 'yc', 'nxc', 'nyc', 'Omega0']

    df = df_doppio.dropna(subset=required).copy()
    print(f'1. Removed {len(df_doppio) - len(df)} eddy-days with NaNs')

    # --- cyclonicity consistency ---
    nenc_cyc = np.where(df['nCyc'].eq('AE'), 1, -1)
    doppio_cyc = np.sign(df['Omega0'].values)

    cyc_mask = nenc_cyc == doppio_cyc
    df = df.loc[cyc_mask].copy()

    print(f'2. Removed {cyc_mask.size - cyc_mask.sum()} eddy-days with cyclonicity mismatch')

    # --- center error ---
    err = np.hypot(
        df['xc'] - df['nxc'],
        df['yc'] - df['nyc']
    )

    df['err'] = err

    # --- domain bounds ---
    boundary_mask = (
        df['xc'].between(0, X_grid.max(), inclusive='neither')
        & df['yc'].between(0, Y_grid.max(), inclusive='neither')
    )

    # --- robust center-error filtering ---
    log_err = np.log1p(err)

    err_med = np.median(log_err)
    err_mad = np.median(np.abs(log_err - err_med))

    if err_mad == 0:
        upper_err = np.inf
    else:
        upper_err = np.expm1(
            err_med + threshold * err_mad / 0.6745
        )

    center_mask = err <= upper_err

    # optional hard cap
    # center_mask &= (err <= 50)

    # --- Omega0 filter ---
    df['Omega0_abs'] = df['Omega0'].abs()

    omega_mask = df['Omega0_abs'] <= Omega0_thresh

    # --- sanity checks ---
    sanity_mask = (
        np.isfinite(err)
        & np.isfinite(df['Omega0_abs'])
        & (df['Omega0_abs'] > 0)
    )

    # --- combined mask ---
    final_mask = (
        boundary_mask
        & center_mask
        & omega_mask
        & sanity_mask
    )

    # --- diagnostics ---
    print('\nThresholds:')
    print(f'Omega0_abs: 0 → {Omega0_thresh} 1/s')
    print(f'Center error: 0 → {upper_err:.3f} km')

    df_clean = (
        df.loc[final_mask]
        .copy()
        .reset_index(drop=True)
    )

    print(f'\n3. Removed {len(df) - len(df_clean)} eddy-days by QC masks')
    print(f'Final retained: {len(df_clean)} eddy-days')

    return df_clean
    

In [5]:
df_clean = clean_doppio(df_doppio, X_grid, Y_grid)


1. Removed 38768 eddy-days with NaNs
2. Removed 682 eddy-days with cyclonicity mismatch

Thresholds:
Omega0_abs: 0 → 5e-05 1/s
Center error: 0 → 23.367 km

3. Removed 9607 eddy-days by QC masks
Final retained: 367194 eddy-days


In [6]:
df_data = df_clean.copy()
df_data = df_data.rename(columns={'nCyc': 'Cyc'})
# df_data['eddy_idx'] = df_data.groupby('Day').cumcount()
df_data['fname'] = df_data['fnumber'].map(
    lambda x: f"/srv/scratch/z3533156/26year_BRAN2020/outer_avg_{int(x):05}.nc"
)
df_data


,Day,fnumber,nxc,nyc,Cyc,xc,yc,w,Q,Omega0,Omega,Rc,psi0,R,err,Omega0_abs,fname
0,1462,1461,830.0,1515.0,AE,829.879126,1515.297884,0.000016,"[[0.7054724746035225, -0.17299518294950286], [...",0.000007,0.000008,96.456316,-38.462785,63.384028,0.321474,0.000007,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
1,1462,1461,358.0,1408.0,AE,357.902634,1407.794571,0.000032,"[[1.2525415454561337, -0.37318304544145586], [...",0.000015,0.000014,78.042405,-41.260426,50.068171,0.227335,0.000015,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
2,1462,1461,928.0,1356.0,CE,928.181752,1356.070550,-0.000011,"[[0.9184334895938384, -0.6568144522261018], [-...",-0.000004,-0.000007,118.032741,47.992205,74.597458,0.194964,0.000004,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
3,1462,1461,506.0,1354.0,CE,505.750855,1353.499933,-0.000032,"[[1.0915204469734063, -0.17483272531676586], [...",-0.000016,-0.000013,106.666574,75.106895,69.457705,0.558695,0.000016,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
4,1462,1461,754.0,1285.0,AE,753.849693,1284.689807,0.000022,"[[1.300245004042572, -0.37537759847059776], [-...",0.000010,0.000008,103.327517,-44.066805,65.961546,0.344691,0.000010,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367189,10650,10641,349.0,158.0,AE,348.663439,158.138527,0.000033,"[[1.3863808771116255, 0.04649055064735413], [0...",0.000016,0.000012,110.552001,-72.997132,43.099927,0.363955,0.000016,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
367190,10650,10641,973.0,126.0,CE,978.294053,127.970950,-0.000004,"[[0.6233322936879349, -0.5502050858849237], [-...",-0.000001,-0.000002,75.127332,4.241051,34.482872,5.649039,0.000001,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
367191,10650,10641,805.0,95.0,AE,804.765913,95.382012,0.000011,"[[1.2619918374118362, 0.3980497594289241], [0....",0.000005,0.000006,60.426625,-10.779905,45.893895,0.448028,0.000005,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
367192,10650,10641,157.0,34.0,CE,157.210669,33.449590,-0.000007,"[[0.9874338034387458, 0.628992620695803], [0.6...",-0.000003,-0.000004,141.142981,36.082072,34.764443,0.589349,0.000003,/srv/scratch/z3533156/26year_BRAN2020/outer_av...


In [7]:
def rotate_uv(u, v, angle):
    u = np.where(np.abs(u) > 1e30, np.nan, u).astype(float)
    v = np.where(np.abs(v) > 1e30, np.nan, v).astype(float)
    u_rot = v * np.sin(angle) + u * np.cos(angle)
    v_rot = v * np.cos(angle) - u * np.sin(angle)

    return u_rot, v_rot
    

In [8]:
def transect_indexer(ic, jc, X, Y, r=30.0):
    x = np.asarray(X[:, 0])
    y = np.asarray(Y[0, :])

    x0, y0 = x[ic], y[jc]

    i0 = np.searchsorted(x, x0 - r, side='left')
    i1 = np.searchsorted(x, x0 + r, side='right')
    j0 = np.searchsorted(y, y0 - r, side='left')
    j1 = np.searchsorted(y, y0 + r, side='right')

    ii = np.arange(i0, i1)
    jj = np.arange(j0, j1)

    return x[ii], np.full(ii.size, y0), np.full(jj.size, x0), y[jj], ii, jj

def nearest_ij(xc, yc, X, Y):
    x = np.asarray(X[:, 0])
    y = np.asarray(Y[0, :])

    ic = np.abs(x - xc).argmin()
    jc = np.abs(y - yc).argmin()

    return int(ic), int(jc)

def interp_3d_to_reference_depths(var3d, z3d, target_depths):
    target_depths = np.asarray(target_depths)
    nz = target_depths.size

    out = np.full(var3d.shape[:2] + (nz,), np.nan)

    for i in range(var3d.shape[0]):
        for j in range(var3d.shape[1]):
            out[i, j, :] = np.interp(
                target_depths,
                z3d[i, j, :],
                var3d[i, j, :],
                left=np.nan,
                right=np.nan
            )

    return out

def local_eddy_arrays(X, Y, u2d, v2d, xc, yc, Q, rho_max):
    local = (
        (X >= xc - rho_max)
        & (X <= xc + rho_max)
        & (Y >= yc - rho_max)
        & (Y <= yc + rho_max)
    )

    Xloc = X[local]
    Yloc = Y[local]
    uloc = u2d[local]
    vloc = v2d[local]

    dx = Xloc - xc
    dy = Yloc - yc

    rho2 = (
        Q[0, 0] * dx**2
        + 2 * Q[0, 1] * dx * dy
        + Q[1, 1] * dy**2
    )

    rho = np.sqrt(np.where(rho2 >= 0, rho2, np.nan))

    return Xloc, Yloc, uloc, vloc, rho

# def vert_doppio_dataset(
#     dic, df_eddies, fname, X_grid, Y_grid, angle, z_r, 
#     r=30.0,
#     max_jump=100,
#     max_depth_levels=None,
#     rho_max=200,
#     rho_min=30,
#     target_depths=None
# ):
#     z_r = np.abs(z_r)

#     df_file = df_eddies.loc[df_eddies.fname.eq(fname)]

#     if df_file.empty:
#         return dic

#     with nc.Dataset(fname) as ds:

#         ocean_time = ds['ocean_time'][:] / 86400
#         time_lookup = {int(d): i for i, d in enumerate(ocean_time)}

#         for day, df_day in df_file.groupby('Day', sort=False):

#             t = time_lookup.get(int(day))
#             if t is None:
#                 continue

#             u3d = ds['u_eastward'][t, :, :, :].T.astype(float)
#             u3d = np.flip(u3d, axis=2)
#             v3d = ds['v_northward'][t, :, :, :].T.astype(float)
#             v3d = np.flip(v3d, axis=2)

#             u3d[np.abs(u3d) > 1e30] = np.nan
#             v3d[np.abs(v3d) > 1e30] = np.nan

#             # nz = u3d.shape[-1]
#             nz = len(target_depths)
#             if max_depth_levels is not None:
#                 nz = min(nz, max_depth_levels)

#             u_depth = interp_3d_to_reference_depths(
#                 u3d, z_r, target_depths
#             )
            
#             v_depth = interp_3d_to_reference_depths(
#                 v3d, z_r, target_depths
#             )

#             for row in df_day.itertuples():

#                 eddy_key = f'Eddy{row.Eddy}'
#                 day_key = f'Day{int(row.Day)}'

#                 dic.setdefault(eddy_key, {})

#                 out = []

#                 out.append({
#                     'z': 0,
#                     'Depth': 0,
#                     'xc': row.xc,
#                     'yc': row.yc,
#                     'ic': row.ic,
#                     'jc': row.jc,
#                     'w': row.w,
#                     'Q': np.array([[row.q11, row.q12], [row.q12, row.q22]]),
#                     'Omega0': np.nan,
#                     'Omega': row.Omega,
#                     'Rc': row.Rc,
#                     'psi0': row.psi0,
#                     'R': row.R
#                 })

#                 xc_prev = row.xc
#                 yc_prev = row.yc
#                 w_surf = row.w

#                 ic = int(row.ic)
#                 jc = int(row.jc)
#                 # print(eddy_key)
#                 for k in range(nz):
#                     # print(f'k = {k}')
            
#                     if (
#                         (xc_prev < r)
#                         or (xc_prev > X_grid.max() - r)
#                         or (yc_prev < r)
#                         or (yc_prev > Y_grid.max() - r)
#                     ):
#                         # print(1)
#                         break

#                     target_depth = target_depths[k]

#                     u2d = u_depth[:, :, k]
#                     v2d = v_depth[:, :, k]

#                     x1, y1, x2, y2, ii, jj = transect_indexer(
#                         ic, jc, X_grid, Y_grid, r=r
#                     )

#                     u1 = u2d[ii, jc]
#                     v1 = v2d[ii, jc]
#                     u2 = u2d[ic, jj]
#                     v2 = v2d[ic, jj]

#                     u1, v1 = rotate_uv(u1, v1, angle)
#                     u2, v2 = rotate_uv(u2, v2, angle)

#                     if any(np.all(np.isnan(a)) for a in [u1, v1, u2, v2]):
#                         # print(2)
#                         break

#                     try:
#                         xc, yc, w, Q, Omega0 = doppio(
#                             x1, y1, u1, v1,
#                             x2, y2, u2, v2
#                         )
#                     except Exception:
#                         print(3)
#                         break

#                     if (
#                         np.sign(w) != np.sign(w_surf)
#                         or np.hypot(xc - xc_prev, yc - yc_prev) > max_jump
#                     ):
#                         # print(4)
#                         break

#                     ic, jc = nearest_ij(xc, yc, X_grid, Y_grid)

#                     u2d_rot, v2d_rot = rotate_uv(u2d, v2d, angle)

#                     Xloc, Yloc, uloc, vloc, rho = local_eddy_arrays(
#                         X_grid, Y_grid,
#                         u2d_rot, v2d_rot,
#                         xc, yc, Q,
#                         rho_max=rho_max
#                     )

#                     mask0 = rho < rho_max

#                     if mask0.sum() < 10:
#                         break

#                     radii = find_directional_radii(
#                         u2d_rot, v2d_rot, X_grid, Y_grid, xc, yc, Q
#                     )

#                     R = np.nanmean([
#                         radii['up'],
#                         radii['right'],
#                         radii['down'],
#                         radii['left'],
#                     ])

#                     if not np.isfinite(R):
#                         # print(5)
#                         break

#                     rho_lim = max(min(R * 1.75, rho_max), rho_min)
#                     mask = rho < rho_lim

#                     if mask.sum() < 10:
#                         break

#                     try:
#                         Rc, psi0, Omega = out_core_param_fit(
#                             Xloc[mask],
#                             Yloc[mask],
#                             uloc[mask],
#                             vloc[mask],
#                             xc, yc, Q,
#                             Omega0=Omega0
#                         )
#                     except Exception:
#                         break

#                     out.append({
#                         'z': k+1,
#                         'Depth': target_depth,
#                         'xc': xc,
#                         'yc': yc,
#                         'ic': ic,
#                         'jc': jc,
#                         'w': w*1e-3,
#                         'Q': Q,
#                         'Omega0': Omega0*1e-3,
#                         'Omega': Omega*1e-3,
#                         'Rc': Rc,
#                         'psi0': psi0,
#                         'R': R
#                     })

#                     xc_prev = xc
#                     yc_prev = yc

#                 dic[eddy_key][day_key] = pd.DataFrame(out)

#     return dic

def add_reaches_50m_flag(
    df_data, X_grid, Y_grid, angle, z_r, target_depths,
    r=30.0, max_jump=100, rho_max=200, rho_min=30,
    depth_thresh=50.0, flag_col='reaches_50m'
):
    z_r = np.abs(z_r)
    df = df_data.copy()
    df[flag_col] = False

    if 'ic' not in df or 'jc' not in df:
        ij = [nearest_ij(xc, yc, X_grid, Y_grid) for xc, yc in zip(df.xc, df.yc)]
        df['ic'], df['jc'] = np.array(ij).T

    target_depths = np.asarray(target_depths)
    kmax = np.where(target_depths >= depth_thresh)[0][0] + 1
    target_check = target_depths[:kmax]

    for fname, df_file in df.groupby('fname', sort=False):
        with nc.Dataset(fname) as ds:
            ocean_time = ds['ocean_time'][:] / 86400
            time_lookup = {int(d): i for i, d in enumerate(ocean_time)}

            for day, df_day in df_file.groupby('Day', sort=False):
                t = time_lookup.get(int(day))
                if t is None:
                    continue

                u3d = np.flip(ds['u_eastward'][t].T.astype(float), axis=2)
                v3d = np.flip(ds['v_northward'][t].T.astype(float), axis=2)

                u3d[np.abs(u3d) > 1e30] = np.nan
                v3d[np.abs(v3d) > 1e30] = np.nan

                u_depth = interp_3d_to_reference_depths(u3d, z_r, target_check)
                v_depth = interp_3d_to_reference_depths(v3d, z_r, target_check)

                for row in df_day.itertuples():
                    xc_prev, yc_prev = row.xc, row.yc
                    ic, jc = int(row.ic), int(row.jc)
                    w_surf = row.w
                    reached = False

                    for k, target_depth in enumerate(target_check):

                        if (
                            (xc_prev < r)
                            or (xc_prev > X_grid.max() - r)
                            or (yc_prev < r)
                            or (yc_prev > Y_grid.max() - r)
                        ):
                            break

                        u2d = u_depth[:, :, k]
                        v2d = v_depth[:, :, k]

                        x1, y1, x2, y2, ii, jj = transect_indexer(
                            ic, jc, X_grid, Y_grid, r=r
                        )

                        u1, v1 = rotate_uv(u2d[ii, jc], v2d[ii, jc], angle)
                        u2, v2 = rotate_uv(u2d[ic, jj], v2d[ic, jj], angle)

                        if any(np.all(np.isnan(a)) for a in [u1, v1, u2, v2]):
                            break

                        try:
                            xc, yc, w, Q, Omega0 = doppio(
                                x1, y1, u1, v1,
                                x2, y2, u2, v2
                            )
                        except Exception:
                            break

                        if (
                            np.sign(w) != np.sign(w_surf)
                            or np.hypot(xc - xc_prev, yc - yc_prev) > max_jump
                        ):
                            break

                        ic, jc = nearest_ij(xc, yc, X_grid, Y_grid)

                        u2d_rot, v2d_rot = rotate_uv(u2d, v2d, angle)

                        Xloc, Yloc, uloc, vloc, rho = local_eddy_arrays(
                            X_grid, Y_grid, u2d_rot, v2d_rot,
                            xc, yc, Q, rho_max=rho_max
                        )

                        if (rho < rho_max).sum() < 10:
                            break

                        radii = find_directional_radii(
                            u2d_rot, v2d_rot, X_grid, Y_grid, xc, yc, Q
                        )

                        R = np.nanmean([
                            radii['up'], radii['right'],
                            radii['down'], radii['left']
                        ])

                        if not np.isfinite(R):
                            break

                        rho_lim = max(min(R * 1.75, rho_max), rho_min)
                        mask = rho < rho_lim

                        if mask.sum() < 10:
                            break

                        try:
                            out_core_param_fit(
                                Xloc[mask], Yloc[mask],
                                uloc[mask], vloc[mask],
                                xc, yc, Q,
                                Omega0=Omega0
                            )
                        except Exception:
                            break

                        if target_depth >= depth_thresh:
                            reached = True
                            break

                        xc_prev, yc_prev = xc, yc

                    df.loc[row.Index, flag_col] = reached
        print(fname)
    return df
    

In [9]:
# # add a flag to df_data 1=the edepth df gets to a depth of at least 50m, and 0 otherwise.
# df_data_small = df_data.copy().iloc[:100]
# target_depths = np.abs(z_r[150,150,1:])

# df_data_checked = add_reaches_50m_flag(
#     df_data_small, X_grid, Y_grid, angle, z_r, target_depths
# )
# df_data_checked


In [10]:
# add a flag to df_data 1=the edepth df gets to a depth of at least 50m, and 0 otherwise.
target_depths = np.abs(z_r[150,150,1:])

df_data_checked = add_reaches_50m_flag(
    df_data, X_grid, Y_grid, angle, z_r, target_depths
)
df_data_checked


KeyboardInterrupt: 